In [1]:
!echo $LD_LIBRARY_PATH


/opt/ros/noetic/lib::/usr/local/cuda-12.4/lib64:/home/liwenbo/CoppeliaSim


In [1]:
import openai
from arguments import get_config
from interfaces import setup_LMP
from visualizers import ValueMapVisualizer
from envs.rlbench_env import VoxPoserRLBench
from utils import set_lmp_objects
import numpy as np
from rlbench import tasks

openai.api_key = "sk-cgpibTHnxWRPOUxdb05uaf9wPc687e0mc4EJIzpNjT3G2q5F"
openai.api_base= "https://api.chatanywhere.tech/v1"

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Setup

In [2]:
config = get_config('rlbench')
# uncomment this if you'd like to change the language model (e.g., for faster speed or lower cost)
# for lmp_name, cfg in config['lmp_config']['lmps'].items():
#     cfg['model'] = 'gpt-3.5-turbo'

# initialize env and voxposer ui
visualizer = ValueMapVisualizer(config['visualizer'])
env = VoxPoserRLBench(visualizer=visualizer)
lmps, lmp_env = setup_LMP(env, config, debug=False)
voxposer_ui = lmps['plan_ui']

##################################################
## voxel resolution: [0.0105 0.0131 0.01  ]
##################################################




## Playground

By default we use one of the instructions that come with each task. However, you may treat each task as simply a scene initialization from RLBench, and feel free to try any task you can come up with for the scene.

Note:
- Whether an instruction can be executed or not depends on 1) whether relevant objects are available, and 2) capabilities of the overall algorithm.
- Each execution may produce one or more visualizations. You may view them in "./visualizations/" folder.
- The prompts are adapted with minimal change from the real-world environment in the VoxPoser paper. If a task failure is due to incorrect code generated by the LLM, feel free to modify the relevant prompt in "./prompts/" folder.
- You may view the reward by printing "env.latest_reward". These are computed by RLBench for each task.
- To inspect in viewer without performing any action, you may call "env.rlbench_env._scene.step()" in a loop.

In [6]:
# # uncomment this to show all available tasks in rlbench
# # NOTE: in order to run a new task, you need to add the list of objects (and their corresponding env names) to the "task_object_names.json" file. See README for more details.
# print([task for task in dir(tasks) if task[0].isupper() and not '_' in task])

In [5]:
# below are the tasks that have object names added to the "task_object_names.json" file
# uncomment one to use
# env.load_task(tasks.PutRubbishInBin)
# env.load_task(tasks.LampOff)
# env.load_task(tasks.OpenWineBottle)
env.load_task(tasks.PushButton)
# env.load_task(tasks.TakeOffWeighingScales)
# env.load_task(tasks.MeatOffGrill)
# env.load_task(tasks.SlideBlockToTarget)
# env.load_task(tasks.TakeLidOffSaucepan)
# env.load_task(tasks.TakeUmbrellaOutOfUmbrellaStand)

descriptions, obs = env.reset()
set_lmp_objects(lmps, env.get_object_names())  # set the object names to be used by voxposer

In [6]:
instruction = np.random.choice(descriptions)
voxposer_ui(instruction)

*** OpenAI API call took 1.83s ***
########################################
## "planner" generated code
## context: "objects = ['button']"
########################################
objects = ['button']
# Query: press the violet button.
composer("move to the center of the button")
composer("push the button by 5cm")
composer("back to default pose")
# done


*** OpenAI API call took 2.03s ***
########################################
## "composer" generated code
########################################
# Query: move to the center of the button.
movable = parse_query_obj('gripper')
affordance_map = get_affordance_map('a point at the center of the button')
execute(movable, affordance_map=affordance_map)


*** OpenAI API call took 1.04s ***
########################################
## "parse_query_obj" generated code
## context: "objects = ['button']"
########################################
objects = ['button']
# Query: gripper.
gripper = detect('gripper')
ret_val = gripper


*** OpenAI API ca